<a href="https://colab.research.google.com/github/Nidhijain0602/CareLoop-Clinics-No-Show-Guard/blob/main/01Ingest_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1.1 — Read the file the foundations way**

In [4]:
import csv
import json

filename =  "Datasets/appointments.csv"

with open(filename, newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    headers = reader.fieldnames
    rows = list(reader)

print("Column names:", headers)
print("Total rows:", len(rows))

Column names: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [5]:

def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])
    return row

accepted = []
rejected = []

for row in rows:
    try:
        converted = parse_row(row.copy())
        accepted.append(converted)
    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))


Accepted rows: 988
Rejected rows: 32


In [8]:

with open("Datasets/rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Saved Datasets/rejected_rows.json")
print("Verification:", len(accepted) + len(rejected) == len(rows))

Saved Datasets/rejected_rows.json
Verification: True


**Task 1.2 — Inspect and clean, pass 1**

In [9]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("File loaded:", filename)
print("Dataset shape:", df.shape)

Saving appointments.csv to appointments.csv
File loaded: appointments.csv
Dataset shape: (1020, 14)


In [10]:

print("DATASET INFORMATION")
df.info()

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())


DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

MISSING VALUES
appointment_id      0
patient_id          0
clinic_id   

In [11]:
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))


Rows after removing duplicates: 1000


In [12]:
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print(df["booking_channel"].value_counts())
print("Unique channels:", df["booking_channel"].nunique())


booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64
Unique channels: 3


In [13]:

df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

df["booked_on"] = pd.to_datetime(df["booked_on"])

print("Missing or invalid appointment dates:",
      df["appt_date"].isna().sum())

Missing or invalid appointment dates: 6


In [14]:

df = df.dropna(subset=["appt_date", "status"])

print("Final rows:", len(df))
print("Missing dates:", df["appt_date"].isna().sum())
print("Missing statuses:", df["status"].isna().sum())

Final rows: 990
Missing dates: 0
Missing statuses: 0


**Task 1.3 — Clean, pass 2: validity, gaps and outliers **

In [15]:

import numpy as np

invalid_age = (df["age"] < 1) | (df["age"] > 100)

print("Invalid ages:", invalid_age.sum())

df.loc[invalid_age, "age"] = np.nan

print("Total missing ages:", df["age"].isna().sum())

Invalid ages: 5
Total missing ages: 17


In [16]:

age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("Median age:", age_median)
print("Median distance:", distance_median)

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Missing ages:", df["age"].isna().sum())
print("Missing distances:", df["distance_km"].isna().sum())


Median age: 48.0
Median distance: 8.25
Missing ages: 0
Missing distances: 0


In [17]:
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    capped_count = (
        (df[col] < lower) | (df[col] > upper)
    ).sum()

    df[col] = np.clip(df[col], lower, upper)

    print(f"\nColumn: {col}")
    print("Q1:", q1)
    print("Q3:", q3)
    print("Upper fence:", upper)
    print("Values capped:", capped_count)



Column: distance_km
Q1: 4.625
Q3: 13.0
Upper fence: 25.5625
Values capped: 26

Column: fee_inr
Q1: 500.0
Q3: 787.5
Upper fence: 1218.75
Values capped: 3


In [18]:


df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Maximum fee:", df["fee_inr"].max())
print("Remaining rows:", len(df))

print("\nRemaining missing values:")
print(df[["age", "distance_km", "fee_inr"]].isna().sum())

Maximum fee: 1219
Remaining rows: 990

Remaining missing values:
age            0
distance_km    0
fee_inr        0
dtype: int64


**Task 1.4 — Features and a NumPy summary **

In [19]:
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

df["no_show"] = (df["status"] == "No-Show").astype(int)

print("Minimum lead days:", df["lead_days"].min())
print("Maximum lead days:", df["lead_days"].max())
print("Total no-shows:", df["no_show"].sum())
print("No-show percentage:",
      round(df["no_show"].mean() * 100, 1))

Minimum lead days: 1
Maximum lead days: 45
Total no-shows: 218
No-show percentage: 22.0


In [20]:


X = df[[
    "age",
    "distance_km",
    "fee_inr",
    "lead_days"
]].to_numpy()

print("Array shape:", X.shape)
print("Column means:", np.mean(X, axis=0))

Array shape: (990, 4)
Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [21]:

mask = X[:, 3] > 14

print("Appointments with lead_days > 14:", mask.sum())
print("Percentage:", round(mask.mean() * 100, 1))

Appointments with lead_days > 14: 296
Percentage: 29.9


In [22]:
distance = X[:, 1]

z_scores = (
    distance - np.mean(distance)
) / np.std(distance)

print("Rows with |z| > 2:",
      (np.abs(z_scores) > 2).sum())

Rows with |z| > 2: 53


**Task 1.5 — Enrich the clinics from a live API**

In [23]:
import pandas as pd
import requests

clinics_filename = "Datasets/clinics.csv"

clinics_df = pd.read_csv(clinics_filename)

print("Clinics shape:", clinics_df.shape)
print(clinics_df.head())

Clinics shape: (8, 6)
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873


In [25]:

import requests
import time

def get_clinic_temperature(row):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    for attempt in range(3):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=30
            )

            response.raise_for_status()

            data = response.json()

            temperatures = data.get(
                "hourly", {}
            ).get("temperature_2m")

            if temperatures:
                return temperatures[0]

            print(f"Missing temperature: {row['clinic_name']}")
            return None

        except (requests.exceptions.RequestException, ValueError) as error:
            print(f"Attempt {attempt + 1} failed: {error}")
            time.sleep(2)

    print(f"Failed: {row['clinic_name']}")
    return None


clinics_df["temp_c"] = clinics_df.apply(
    get_clinic_temperature,
    axis=1
)

print(clinics_df[["clinic_name", "temp_c"]])


                clinic_name  temp_c
0     Kothrud Family Clinic    16.0
1      Hadapsar Care Centre    17.3
2    Sitabuldi Health Point    20.8
3      Malviya Nagar Clinic    15.8
4     Vaishali Nagar Clinic    15.8
5  Edappally Medical Centre    25.8
6        Vijay Nagar Clinic    17.4
7    Gomti Nagar Care Point    17.7


In [26]:

clinics_df.to_csv(
    "clinics_enriched.csv",
    index=False
)

print("File saved: clinics_enriched.csv")
print("Rows:", len(clinics_df))
print("Columns:", len(clinics_df.columns))
print("Missing temperatures:",
      clinics_df["temp_c"].isna().sum())

File saved: clinics_enriched.csv
Rows: 8
Columns: 7
Missing temperatures: 0


In [27]:
columns = [
    "appointment_id", "patient_id", "clinic_id",
    "booked_on", "appt_date", "age", "gender",
    "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows",
    "status", "lead_days", "no_show"
]

print("Unique appointment IDs:",
      df["appointment_id"].nunique())

print("Duplicate appointment IDs:",
      df["appointment_id"].duplicated().sum())

print("Missing values:")
print(df[columns].isna().sum())

assert df["appointment_id"].is_unique
assert df[columns].isna().sum().sum() == 0
assert len(df) == 990


Unique appointment IDs: 990
Duplicate appointment IDs: 0
Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64


In [28]:
df = df.sort_values("appointment_id").copy()

df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

df = df[columns]

df.to_csv("appointments_clean.csv", index=False)

print("Saved appointments_clean.csv")
print("Shape:", df.shape)


Saved appointments_clean.csv
Shape: (990, 16)


**Task 1.6 — Export the clean table and write the SQL load script **

In [30]:

def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, (str,)):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

with open("Datasets/load_data.sql", "w", encoding="utf-8") as file:

    for _, row in clinics_df.iterrows():
        values = ", ".join(
            sql_value(row[col])
            for col in clinics_df.columns
        )
        file.write(f"INSERT INTO clinics VALUES ({values});\n")

    for _, row in df.iterrows():
        values = ", ".join(
            sql_value(row[col])
            for col in columns
        )
        file.write(f"INSERT INTO appointments VALUES ({values});\n")

print("Saved load_data.sql")


Saved load_data.sql


In [31]:
with open("load_data.sql", "r", encoding="utf-8") as file:
    statements = [
        line for line in file
        if line.startswith("INSERT INTO")
    ]

print("Total INSERT statements:", len(statements))
print("Clinic INSERT statements:",
      sum("INSERT INTO clinics" in s for s in statements))
print("Appointment INSERT statements:",
      sum("INSERT INTO appointments" in s for s in statements))

print("First statement:", statements[0][:100])
print("Last statement:", statements[-1][:100])

assert len(statements) == 998

Total INSERT statements: 998
Clinic INSERT statements: 8
Appointment INSERT statements: 990
First statement: INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 16.0)
Last statement: INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 


In [32]:
from google.colab import files

files.download("appointments_clean.csv")
files.download("load_data.sql")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:

clinic_summary = (
    df.groupby("clinic_id")
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("no_show", "sum")
      )
      .reset_index()
)

clinic_summary = clinic_summary.merge(
    clinics_df[["clinic_id", "clinic_name", "city"]],
    on="clinic_id"
)

clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"]
    / clinic_summary["appointments"]
).round(1)

print(
    clinic_summary[
        ["clinic_name", "city", "appointments",
         "no_shows", "no_show_pct"]
    ].sort_values("no_show_pct", ascending=False)
)

                clinic_name     city  appointments  no_shows  no_show_pct
7    Gomti Nagar Care Point  Lucknow            90        30         33.3
1      Hadapsar Care Centre     Pune           143        39         27.3
3      Malviya Nagar Clinic   Jaipur           142        38         26.8
0     Kothrud Family Clinic     Pune           160        36         22.5
6        Vijay Nagar Clinic   Indore           101        21         20.8
5  Edappally Medical Centre    Kochi           142        23         16.2
2    Sitabuldi Health Point   Nagpur            82        13         15.9
4     Vaishali Nagar Clinic   Jaipur           130        18         13.8
